# 02 GPU、显存与计算基础

![GPU memory concept](../assets/images/gpu_memory_concept_unlabeled.png)

AI Infra 里最常见的资源瓶颈是 GPU。对产品/研发来说，不需要掌握硬件微架构，但必须理解 GPU 为什么适合 LLM、显存里装了什么、dtype 和量化为什么重要、为什么长上下文和高并发会让服务崩掉。


## 1. CPU 和 GPU 的分工

CPU 擅长复杂控制流、系统调用、网络、业务逻辑；GPU 擅长大量相似的并行计算。LLM 推理主体是矩阵乘法和 attention，因此适合 GPU；但请求调度、tokenization、HTTP、日志、鉴权仍然需要 CPU。

```mermaid
flowchart LR
    Req["HTTP 请求"] --> CPU["CPU<br/>鉴权 / tokenization / 调度"]
    CPU --> Batch["组 batch<br/>tensor 工作"]
    Batch --> GPU["GPU/Metal<br/>矩阵乘法 / attention"]
    GPU --> VRAM["显存<br/>weights / KV cache / activations"]
    GPU --> CPU
    CPU --> Resp["流式响应"]
```

所以 GPU 利用率低不一定是 GPU 太弱，也可能是 CPU/tokenizer/调度/网络没有把工作喂满。


## 2. 显存里到底有什么

推理时显存通常包括：

| 类别 | 是什么 | 和什么相关 |
|---|---|---|
| 模型权重 weights | 已训练好的参数矩阵 | 参数量、dtype、量化方式 |
| KV cache | 历史 token 的 Key/Value 缓存 | 层数、KV heads、上下文长度、并发 |
| 激活 activations | 当前 forward 的中间结果 | batch token 数、模型结构 |
| workspace/allocator | runtime 预留、kernel 临时 buffer、碎片 | 框架和后端实现 |

粗略权重显存公式：

$$
\text{weight\_memory} \approx \text{parameters} \times \text{bytes\_per\_parameter}
$$

KV cache 公式：

$$
\text{KV bytes} \approx L_{layers} \times 2 \times B \times S \times H_{kv} \times D_{head} \times bytes_{dtype}
$$

其中 `2` 表示 Key 和 Value 两份缓存，`B` 是并发序列数，`S` 是上下文长度。


In [ ]:
def gib(bytes_count):
    return bytes_count / (1024 ** 3)

def weight_memory_gib(params_billion: float, bytes_per_param: float) -> float:
    return gib(params_billion * 1e9 * bytes_per_param)

def kv_cache_gib(layers, batch, seq_len, kv_heads, head_dim, dtype_bytes=2):
    return gib(layers * 2 * batch * seq_len * kv_heads * head_dim * dtype_bytes)

print('权重显存估算')
for params in [1.5, 7, 14, 70]:
    print(f'\n{params}B parameters')
    for name, b in {'FP16/BF16': 2, 'INT8': 1, 'INT4': 0.5}.items():
        print(f'  {name:9s}: {weight_memory_gib(params, b):6.2f} GiB')

print('\nKV cache 估算：32 layers, 8 kv heads, head_dim=128, FP16')
for seq_len in [2048, 8192, 32768]:
    for batch in [1, 8, 32]:
        print(f'seq={seq_len:5d}, batch={batch:2d} -> KV ≈ {kv_cache_gib(32, batch, seq_len, 8, 128):6.2f} GiB')


## 3. dtype 与量化

dtype 决定每个数占多少字节，也影响速度、精度和硬件支持。

| dtype | 字节/参数 | 常见用途 | 风险 |
|---|---:|---|---|
| FP32 | 4 | 训练、精度基线 | 推理太占显存 |
| FP16/BF16 | 2 | GPU 推理主流 | 需要硬件支持 |
| FP8 | 1 | 新 GPU 上的高性能推理 | 兼容性和校准要求 |
| INT8 | 1 | 量化推理 | 可能损失质量 |
| INT4 | 0.5 | 本地/低成本推理 | 质量损失和 kernel 支持更敏感 |

量化不是免费午餐。它能降低权重显存和带宽压力，但可能引入质量下降，也可能受限于推理框架、硬件和模型结构。产品上常见做法是：先用 FP16/BF16 做质量基准，再评估 INT8/INT4 是否能接受。


## 4. 算力、带宽和利用率

LLM 推理不总是“算力越高越快”。decode 阶段每次生成一个 token，需要读取大量权重和 KV cache，常常受到 memory bandwidth 影响。prefill 阶段更像大矩阵计算，通常更能吃满 GPU。

```mermaid
flowchart TB
    Workload["工作负载 / workload"] --> Prompt["长 prompt<br/>prefill heavy"]
    Workload --> Decode["长输出/高并发<br/>decode heavy"]
    Prompt --> Compute["更依赖算力和 batch token budget"]
    Decode --> Bandwidth["更依赖显存带宽和 KV cache 读取"]
    Compute --> Metrics["首 token 与吞吐 / TTFT / throughput"]
    Bandwidth --> Metrics["输出间隔与速度 / ITL / tokens/sec"]
```

这也是为什么调优时要区分 TTFT 和 ITL：它们背后的瓶颈可能不同。


In [ ]:
# 用简单模型理解 batch token budget 对吞吐的影响。
# 假设每次调度有固定开销，并且每个 token 有计算成本。
def simulate_step_time(tokens, fixed_ms=2.0, per_token_ms=0.05):
    return fixed_ms + tokens * per_token_ms

for tokens in [128, 512, 2048, 8192]:
    ms = simulate_step_time(tokens)
    throughput = tokens / (ms / 1000)
    print(f'batch_tokens={tokens:5d} step_time={ms:7.1f}ms throughput≈{throughput:9.0f} tokens/s')


## 5. 常见硬件的产品化直觉

- **H100/A100**：高端数据中心 GPU，适合大模型、高吞吐、多 GPU 并行和严肃生产服务。
- **L4/T4**：更偏成本友好和中小模型服务，适合轻量 API、embedding、低中等并发。
- **消费级 GPU**：适合实验，但生产运维、显存、稳定性、驱动和多租户能力有限。
- **Apple Silicon/Metal**：适合本地学习、小模型、隐私和开发体验；不要把它等同于 CUDA 生产主路径。
- **CPU**：适合小模型、低 QPS、离线任务或 fallback，但大模型交互式生成通常成本/延迟不理想。

产品判断不是“哪个卡最强”，而是 workload、SLA、预算、团队运维能力和供应链的折中。


## 6. 常见误区

- **只按权重估算显存。** KV cache 往往是长上下文和高并发服务的真正瓶颈。
- **GPU 利用率越高越好。** 利用率高但 p99 延迟爆炸，不是好服务。
- **量化一定更好。** 量化可能降低质量，也可能因为 kernel 不匹配导致速度没有预期提升。
- **Mac 跑得动就能生产部署。** Mac 本地适合学习和开发，不代表生产吞吐和稳定性。


## 7. Batch 在训练和推理中的含义不同

训练里的 batch size 通常表示一次用多少样本计算梯度。推理里的 batch 更动态：服务端会把多个用户请求在某个调度 step 中组合起来，以提高 GPU 利用率。LLM serving 还常用 `batch tokens` 而不是简单的 `batch requests`，因为请求长度差异巨大。

例如：8 个短 prompt 和 1 个超长 prompt 的计算成本可能完全不同。生产调优时要关注：

- 同时运行的 requests 数量。
- 每个 step 的 prefill tokens 和 decode tokens。
- `max_num_batched_tokens` 或类似 token budget 参数。
- 长 prompt 是否阻塞 decode。

这就是为什么 “QPS=10” 对 LLM 服务来说信息不足。你还需要 token 长度分布。


In [ ]:
# 同样 10 QPS，不同 token 分布会造成完全不同的 token pressure。
workloads = {
    'short_chat': {'qps': 10, 'prompt': 300, 'output': 120},
    'long_rag': {'qps': 10, 'prompt': 6000, 'output': 250},
    'agent': {'qps': 10, 'prompt': 1800, 'output': 600},
}
for name, w in workloads.items():
    prompt_rate = w['qps'] * w['prompt']
    output_rate = w['qps'] * w['output']
    print(f'{name:10s} prompt_tokens/s={prompt_rate:6d}, output_tokens/s={output_rate:5d}')


## 8. GPU 利用率怎么看

GPU 利用率不是一个万能指标。你可能看到：

- GPU utilization 很低：可能 batch 太小、CPU/tokenizer 慢、请求不足、网络或调度瓶颈。
- GPU utilization 很高但延迟差：可能过度追求吞吐，queue time 或 ITL 已经不可接受。
- 显存占用很高但计算利用率低：可能模型权重和 KV cache 占满显存，但请求不足或 decode 受带宽限制。

对 LLM 服务，更实用的 dashboard 应该同时看：GPU utilization、memory used、KV cache usage、request queue time、TTFT、ITL、tokens/sec、error/OOM/preemption。


## 9. 面试/工作表达

> 我做 GPU 容量判断时不会只看模型参数量。权重显存只是基础，还要估 KV cache：层数、并发、上下文长度、KV heads、head dim 和 dtype 都会影响显存。推理性能还要区分 prefill 和 decode：prefill 更像大矩阵计算，decode 更容易受 KV cache 读取和内存带宽影响。所以我会结合 token 分布、TTFT/ITL、KV cache 使用率和真实 benchmark 做判断。


## 10. 从显存预算到产品取舍

假设你要在一张 24GB 显存的卡上部署 7B 模型。FP16 权重大约 13GB，看起来还剩 11GB，但这不代表可以放心服务长上下文高并发。你还要留出 KV cache、激活、runtime workspace 和安全余量。如果启用 8k 或 32k 上下文，并发再上来，KV cache 很快会吃掉剩余显存。

这时有几类选择：第一，降低模型大小或使用量化，减少权重显存；第二，降低 `max_model_len` 或限制长 prompt，减少 KV cache；第三，降低并发或增加副本，减少单实例压力；第四，优化 RAG 检索和上下文压缩，从源头减少 prompt tokens；第五，换更大显存或更高带宽 GPU。每个选择都有产品影响：小模型可能质量下降，短上下文可能回答不了复杂问题，加 GPU 会增加成本，压缩上下文可能丢信息。

这就是产品/研发需要懂 GPU 基础的原因：很多“模型效果问题”最后都会变成资源、成本和体验的取舍。


## 11. GPU 资源问题的排查路径

当线上服务变慢或 OOM，可以按这个顺序排查：

1. 看请求 token 分布：是否出现超长 prompt 或超长 output？
2. 看 queue time：是否请求太多，实例来不及处理？
3. 看 KV cache usage：是否接近上限，是否出现 preemption？
4. 看 GPU memory：是否权重、KV cache、workspace 已经占满？
5. 看 GPU utilization：是没有喂满，还是已经过载？
6. 看 CPU/tokenizer：是否 CPU 预处理成为瓶颈？
7. 看错误：OOM、timeout、driver reset、container restart 是否出现？

不要只盯 `nvidia-smi` 的一个数字。LLM 服务是多指标系统，显存、利用率、队列、token rate、TTFT、ITL 要一起看。


## 12. 自测题

1. 为什么 7B FP16 模型不能简单理解为“只需要 14GB 显存”？
2. INT4 量化降低了什么成本？可能带来什么风险？
3. 为什么 decode 阶段可能更受内存带宽影响？
4. 同样 QPS 下，短 chat 和长 RAG 哪个更容易造成显存压力？
5. Mac Metal 本地跑通模型，为什么不能直接说明生产环境可行？


## 13. 一个显存预算例题

假设你要部署 14B 模型，使用 FP16，目标支持 8k 上下文，单实例同时服务 8 个活跃请求。权重显存粗估为 `14B * 2 bytes = 28GB`，换算约 26GiB。再估 KV cache：如果模型 40 层、8 个 KV heads、head_dim 128、FP16，那么 KV cache 约为 `40 * 2 * 8 * 8192 * 8 * 128 * 2 bytes`，约 10GiB 级别。再加 activations、workspace、安全余量，一张 40GB 卡可能很紧张。

此时你有几种方案：换 4bit/8bit 权重量化，降低 `max_model_len`，降低单实例并发，换更大显存 GPU，或者从 RAG 侧减少 prompt。真正的工程判断不是“14B 能不能跑”，而是“在目标 SLA、并发和 token 分布下是否稳定”。

这类估算在需求评审时非常有用。它能帮助你提前发现：产品想要 32k 上下文、高并发、低成本、小 GPU，可能是不可能三角。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
